# Lab: The BB84 Quantum Key Distribution Protocol with Qiskit

In this lab we implement **BB84**, the first quantum key distribution (QKD) protocol, introduced by Bennett and Brassard in 1984.

Alice and Bob want to agree on a shared secret key. Alice can send individual qubits to Bob over a **quantum channel**, and Alice and Bob can also talk over a **public classical channel**. An eavesdropper, Eve, may listen to both channels, but she cannot clone unknown quantum states.

BB84 uses two bases:

| Basis | Bit 0 | Bit 1 |
|:---:|:---:|:---:|
| $Z$ (computational) | $|0\rangle$ | $|1\rangle$ |
| $X$ (Hadamard) | $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ | $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$ |

Two states from different bases are not orthogonal, so measuring in the wrong basis gives a random result. This is the source of BB84's security.

The protocol is:

1. Alice chooses a random bit and a random basis for each qubit, encodes the bit, and sends the qubit to Bob.
2. Bob chooses a random basis for each qubit and measures it.
3. Alice and Bob publicly announce their **bases** (but not their bits) and keep only the positions where their bases match. This is called **sifting**.
4. In the absence of noise and eavesdropping, the sifted bits agree exactly and form a shared key.

We will build the protocol in four tasks:
1. Encode a single bit in a chosen basis.
2. Measure qubits in a chosen basis.
3. Sift Alice's and Bob's raw bits into their keys.
4. Run the full protocol end to end.

We then add an eavesdropper and implement the security check Alice and Bob use to detect her.

In [1]:
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

We represent a bitstring such as `"0110"` as a Python string, and a sequence of bases such as `"zxzx"` as a string using `'z'` and `'x'`.

The helpers below validate inputs, generate random bits and bases, and are used by the test cells:

- `validate_bits(bits)` and `validate_bases(bases, n)` check that strings contain only valid characters.
- `validate_bits_and_bases(bits, bases)` checks that the two strings have the same length.
- `random_bits(n, rng)` and `random_bases(n, rng)` draw random bitstrings and basis strings from a NumPy random generator.
- `encode_message(bits, bases)` builds the full encoding circuit for Alice by calling your `encode_bit` function from Task 1.

In [2]:
def validate_bits(bits: str, name: str = "bits") -> None:
    """Guardrail: checks that bits is a non-empty binary string."""
    assert isinstance(bits, str), f"{name} must be a string."
    assert len(bits) >= 1, f"{name} must have at least one bit."
    assert set(bits).issubset({'0', '1'}), (
        f"{name} '{bits}' must only contain '0' and '1'."
    )


def validate_bases(bases: str, n: int | None = None) -> None:
    """Guardrail: checks that bases is a non-empty string over {'z', 'x'}."""
    assert isinstance(bases, str), "Bases must be a string."
    assert len(bases) >= 1, "Bases must have at least one entry."
    assert set(bases).issubset({'z', 'x'}), (
        f"Bases '{bases}' must only contain 'z' and 'x'."
    )
    if n is not None:
        assert len(bases) == n, (
            f"Expected {n} bases, but got {len(bases)}."
        )


def validate_bits_and_bases(bits: str, bases: str) -> None:
    """Guardrail: checks bits and bases are valid and have equal length."""
    validate_bits(bits)
    validate_bases(bases, len(bits))


def random_bits(n: int, rng: np.random.Generator) -> str:
    """Returns a random n-bit string."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    return ''.join(rng.choice(['0', '1'], size=n))


def random_bases(n: int, rng: np.random.Generator) -> str:
    """Returns a random string of n bases, each 'z' or 'x'."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    return ''.join(rng.choice(['z', 'x'], size=n))


def encode_message(bits: str, bases: str) -> QuantumCircuit:
    """
    Encodes a full message by calling encode_bit on each bit and basis.

    The returned circuit has one qubit per bit, with qubit i holding bits[i]
    encoded in bases[i].
    """
    validate_bits_and_bases(bits, bases)

    circuit = QuantumCircuit(len(bits))
    for i, (bit, basis) in enumerate(zip(bits, bases)):
        encode_bit(circuit, bit, basis, i)
    return circuit


print("Helper functions and guardrails loaded successfully!")

Helper functions and guardrails loaded successfully!


## Task 1: Encoding a Bit in a Basis

Alice encodes each bit in the basis she chose:

- In the $Z$ basis, bit `0` is $|0\rangle$ and bit `1` is $|1\rangle$.
- In the $X$ basis, bit `0` is $|+\rangle$ and bit `1` is $|-\rangle$.

Starting from $|0\rangle$, an $X$ gate prepares $|1\rangle$. Applying a Hadamard gate afterwards prepares the $X$-basis states:

$$H|0\rangle = |+\rangle, \qquad H|1\rangle = |-\rangle.$$

### Task 1.1: Implement the Encoder

Implement `encode_bit(circuit: QuantumCircuit, bit: str, basis: str, index: int) -> QuantumCircuit`.

- `circuit`: the circuit to modify.
- `bit`: a single character, `'0'` or `'1'`.
- `basis`: a single character, `'z'` or `'x'`.
- `index`: the index of the qubit in `circuit` to encode.
- Returns the same circuit with the encoding gates applied to qubit `index`.

> Hint: apply an $X$ gate to qubit `index` when `bit == '1'`, then apply a Hadamard gate to qubit `index` when `basis == 'x'`.

In [5]:
def encode_bit(
    circuit: QuantumCircuit,
    bit: str,
    basis: str,
    index: int,
) -> QuantumCircuit:
    """
    Applies the encoding of a single bit in the given basis to one qubit.

    Args:
        circuit: The circuit to modify.
        bit: A single character, '0' or '1'.
        basis: A single character, 'z' or 'x'.
        index: The index of the qubit in circuit to encode.

    Returns:
        QuantumCircuit: The same circuit with the encoding gates applied to qubit index.
    """
    validate_bits(bit, "bit")
    assert len(bit) == 1, "bit must be a single character."
    validate_bases(basis, 1)
    assert isinstance(index, int), "index must be an integer."
    assert 0 <= index < circuit.num_qubits, "index is out of range for the circuit."
    assert circuit.num_clbits == 0, "circuit must not have measurements."

    # TODO: apply the encoding gates to the qubit at the given index
    n = circuit.num_qubits
    # print(n)

    if bit == '1':
        circuit.x(index)
    if basis == 'x':
        circuit.h(index)

    return circuit

### Task 1.2: Test the Encoder

The cell below compares your circuit with the expected statevector for all four bit/basis combinations.

In [6]:
expected_states = {
    ('0', 'z'): np.array([1, 0]),
    ('1', 'z'): np.array([0, 1]),
    ('0', 'x'): np.array([1, 1]) / np.sqrt(2),
    ('1', 'x'): np.array([1, -1]) / np.sqrt(2),
}

for (bit, basis), expected in expected_states.items():
    circuit = QuantumCircuit(1)
    encoded = encode_bit(circuit, bit, basis, 0)
    assert encoded is circuit, "encode_bit must return the circuit it was given."
    state = Statevector(circuit)
    assert state.equiv(Statevector(expected)), (
        f"Encoding bit={bit} in basis={basis} produced the wrong state."
    )

print("Encoder passed all tests!")

Encoder passed all tests!


## Task 2: Measuring in a Basis

To read a qubit, Bob must measure in the basis he chose. Qiskit measurements are always in the $Z$ basis, so measuring in the $X$ basis means applying a Hadamard gate first:

$$H|+\rangle = |0\rangle, \qquad H|-\rangle = |1\rangle.$$

Because $H$ is its own inverse, this is the same gate Alice used to prepare the $X$-basis states. If Bob chooses the same basis as Alice, he recovers her bit with certainty. If he chooses the other basis, his outcome is random.

### Task 2.1: Implement the Measurement

Implement `measure_in_basis(circuit: QuantumCircuit, bases: str) -> str`.

- `circuit`: an `n`-qubit circuit. It may already contain measurements, as it will when Bob receives qubits from Eve.
- `bases`: a string of `n` bases, one per qubit.
- Returns the measured bitstring of length `n`.

The returned string is ordered so that character `i` is the outcome for qubit `i`. Qiskit's `get_counts()` displays outcomes with the highest-index qubit first, so remember to reverse the count string.

> Hint: copy the circuit, apply an $H$ gate to every qubit whose basis is `'x'`, call `measure_all()`, and run on `AerSimulator(method='stabilizer')` with `shots=1`. BB84 only uses Clifford gates, so the stabilizer simulator is exact and handles many qubits efficiently. `measure_all()` adds a new classical register; if the circuit already has measurements, that new register appears first in the counts key, so split the key on whitespace and use the first part before reversing it.

In [ ]:
def measure_in_basis(circuit: QuantumCircuit, bases: str) -> str:
    """
    Measures every qubit of a circuit in the given basis.

    Args:
        circuit: An n-qubit circuit. It may already contain measurements.
        bases: A string of n bases, one per qubit.

    Returns:
        str: The measured bitstring, where character i is the outcome for qubit i.
    """
    validate_bases(bases)
    assert circuit.num_qubits == len(bases), (
        f"Expected {len(bases)} qubits, but the circuit has {circuit.num_qubits}."
    )

    # TODO: copy the circuit, change basis where needed, measure, and run it
    pass

### Task 2.2: Test the Measurement

The cell below prepares every single-bit state and checks that measuring in the **same** basis recovers the bit. It also checks a multi-qubit message and verifies that the input circuit is left unchanged.

In [ ]:
for bit in ['0', '1']:
    for basis in ['z', 'x']:
        circuit = QuantumCircuit(1)
        encode_bit(circuit, bit, basis, 0)
        assert measure_in_basis(circuit, basis) == bit, (
            f"Measuring bit={bit} in basis={basis} did not recover the bit."
        )

rng = np.random.default_rng(1234)
for n in [2, 5, 8]:
    alice_bits = random_bits(n, rng)
    alice_bases = random_bases(n, rng)
    circuit = encode_message(alice_bits, alice_bases)

    assert measure_in_basis(circuit, alice_bases) == alice_bits, (
        f"Same-basis measurement failed for bits={alice_bits}, bases={alice_bases}."
    )
    assert circuit.num_clbits == 0, (
        "measure_in_basis must not add measurements to the input circuit."
    )

print("Measurement passed all tests!")

## Task 3: Sifting the Raw Bits

After Bob has measured every qubit, he and Alice publicly compare their **bases**. They discard every position where the bases differ. The remaining bits are called the **sifted key**.

For example, with

- Alice's bases: `z z x x`
- Bob's bases:   `z x z x`

only positions 0 and 3 are kept. If Alice sent bits `0 1 0 1`, both keys become `0 1`.

Alice and Bob keep their own bitstrings separate: they compare bases, not bits, so Eve learns nothing about the key from the sifting step.

### Task 3.1: Implement Sifting

Implement `sift_key(alice_bits, alice_bases, bob_bits, bob_bases) -> tuple[str, str]`.

- All four arguments are strings of the same length.
- Returns `(alice_key, bob_key)`, keeping only the positions where `alice_bases[i] == bob_bases[i]`.

> Hint: use `zip` to walk over all four strings at once.

In [ ]:
def sift_key(
    alice_bits: str,
    alice_bases: str,
    bob_bits: str,
    bob_bases: str,
) -> tuple[str, str]:
    """
    Sifts Alice's and Bob's raw bits down to the positions where their bases match.

    Args:
        alice_bits: Alice's raw bitstring.
        alice_bases: Alice's basis choices.
        bob_bits: Bob's measured bitstring.
        bob_bases: Bob's basis choices.

    Returns:
        tuple[str, str]: Alice's sifted key and Bob's sifted key.
    """
    validate_bits_and_bases(alice_bits, alice_bases)
    validate_bits_and_bases(bob_bits, bob_bases)
    assert len(alice_bits) == len(bob_bits), (
        "Alice's and Bob's bitstrings must have the same length."
    )

    # TODO: keep only the positions where Alice's and Bob's bases match
    pass

### Task 3.2: Test Sifting

The cell below checks several fixed cases, including one with no matching bases.

In [ ]:
CASES = [
    ("0101", "zzxx", "0101", "zxzx", "01", "01"),
    ("101", "zxz", "010", "zxz", "101", "010"),
    ("0011", "xxxx", "1100", "xxzx", "001", "110"),
    ("01", "zx", "10", "xz", "", ""),
]

for alice_bits, alice_bases, bob_bits, bob_bases, expected_a, expected_b in CASES:
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    assert (alice_key, bob_key) == (expected_a, expected_b), (
        f"Sifting failed for bases={alice_bases} and {bob_bases}: "
        f"got ({alice_key!r}, {bob_key!r}), expected ({expected_a!r}, {expected_b!r})"
    )

print("Sifting passed all tests!")

## Task 4: Running the Full Protocol

Now put the pieces together. Given Alice's bits, Alice's bases, and Bob's bases, the full protocol:

1. Encodes Alice's message with `encode_message`.
2. Lets Bob measure every qubit with `measure_in_basis`.
3. Sifts both raw bitstrings with `sift_key`.

In an honest run (no noise and no Eve), the two sifted keys are identical.

### Task 4.1: Implement BB84

Implement `bb84(alice_bits, alice_bases, bob_bases) -> tuple[str, str]`.

- `alice_bits`: the bits Alice wants to send.
- `alice_bases`: Alice's basis choices, one per bit.
- `bob_bases`: Bob's basis choices, one per bit.
- Returns `(alice_key, bob_key)` after sifting.

> Hint: build the circuit, measure it, and sift the results using the functions you implemented above.

In [ ]:
def bb84(
    alice_bits: str,
    alice_bases: str,
    bob_bases: str,
) -> tuple[str, str]:
    """
    Runs the BB84 protocol for the given bits and basis choices.

    Args:
        alice_bits: Alice's raw bitstring.
        alice_bases: Alice's basis choices, one per bit.
        bob_bases: Bob's basis choices, one per bit.

    Returns:
        tuple[str, str]: Alice's and Bob's sifted keys.
    """
    validate_bits_and_bases(alice_bits, alice_bases)
    validate_bases(bob_bases, len(alice_bits))

    # TODO: encode, measure with Bob's bases, and sift the results
    pass

### Final Test

The cell below runs BB84 on many random inputs. For every input, the two sifted keys must match exactly and must equal the bits on the positions where Alice's and Bob's bases agree. It must pass for the lab to be complete.

In [ ]:
rng = np.random.default_rng(2024)

for n in [1, 2, 3, 5, 8, 13]:
    for _ in range(10):
        alice_bits = random_bits(n, rng)
        alice_bases = random_bases(n, rng)
        bob_bases = random_bases(n, rng)

        expected_key = ''.join(
            bit
            for bit, alice_basis, bob_basis in zip(alice_bits, alice_bases, bob_bases)
            if alice_basis == bob_basis
        )

        alice_key, bob_key = bb84(alice_bits, alice_bases, bob_bases)
        assert alice_key == expected_key, (
            f"Alice's key was wrong for bits={alice_bits}, "
            f"alice_bases={alice_bases}, bob_bases={bob_bases}: "
            f"got {alice_key!r}, expected {expected_key!r}"
        )
        assert bob_key == expected_key, (
            f"Bob's key was wrong for bits={alice_bits}, "
            f"alice_bases={alice_bases}, bob_bases={bob_bases}: "
            f"got {bob_key!r}, expected {expected_key!r}"
        )

print("BB84 passed all tests!")

## Detecting an Eavesdropper

BB84 is secure because Eve cannot learn a qubit's basis without risking a detectable disturbance. In an **intercept-resend** attack, Eve intercepts every qubit, measures it in a basis of her choosing, and forwards it to Bob. The measurement leaves the qubit in the state corresponding to Eve's outcome.

When Eve's basis matches Alice's, she learns the bit and Bob receives the correct state. When it does not, Eve still gets the correct bit with probability $1/2$, but the state she forwards is not the one Alice prepared. Bob's result is then random even when his basis matches Alice's.

On the positions that survive sifting, this produces an expected error rate of about **25%**. Alice and Bob can sacrifice part of the sifted key to estimate the **quantum bit error rate (QBER)**. If the QBER is above a small threshold, they conclude that Eve was listening and abort the key.

### Task 5.1: Implement the Attack

Implement `eve_intercept_resend(circuit: QuantumCircuit, eve_bases: str) -> QuantumCircuit`.

- `circuit`: Alice's encoded circuit, with no measurements.
- `eve_bases`: Eve's basis choices, one per qubit.
- Returns the same circuit with Eve's measurements added. The qubits are left in the post-measurement states that Eve forwards to Bob.

> Hint: copy the circuit and add a classical register for Eve's outcomes. For every qubit whose Eve basis is `'x'`, apply an $H$ gate before measuring, then apply another $H$ after measuring to undo the basis change. Measure every qubit into Eve's register. The second $H$ does not change Eve's recorded bit; it leaves the qubit in the $X$-basis state she measured. Do not prepare a new circuit; Bob will measure the same qubits afterwards.

In [ ]:
def eve_intercept_resend(
    circuit: QuantumCircuit,
    eve_bases: str,
) -> QuantumCircuit:
    """
    Simulates an intercept-resend attack.

    Eve measures each qubit in eve_bases[i] and forwards the qubit in the
    measured state to Bob.

    Args:
        circuit: Alice's encoded circuit, with no measurements.
        eve_bases: Eve's basis choices, one per qubit.

    Returns:
        QuantumCircuit: The same circuit with Eve's measurements added.
    """
    validate_bases(eve_bases, circuit.num_qubits)
    assert circuit.num_clbits == 0, "The input circuit must not have measurements."

    # TODO: measure every qubit in Eve's basis and leave the qubits in the
    # post-measurement states that she forwards to Bob
    pass

### Task 5.2: Measure the Error Rate

Run the cell below to simulate the attack on five batches of 300 qubits and estimate the QBER. In an honest run it would be zero; with Eve listening, it should be close to 25%.

In [ ]:
rng = np.random.default_rng(7)
trials = 5
n = 300
total_sifted = 0
total_errors = 0

for _ in range(trials):
    alice_bits = random_bits(n, rng)
    alice_bases = random_bases(n, rng)
    bob_bases = random_bases(n, rng)
    eve_bases = random_bases(n, rng)

    circuit = encode_message(alice_bits, alice_bases)
    eve_circuit = eve_intercept_resend(circuit, eve_bases)
    bob_bits = measure_in_basis(eve_circuit, bob_bases)
    alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)

    total_sifted += len(alice_key)
    total_errors += sum(a != b for a, b in zip(alice_key, bob_key))

print(f"Total sifted bits: {total_sifted}")
print(f"Errors: {total_errors} ({total_errors / total_sifted:.1%})")

## Task 6: Security Checking and Retrying

Alice and Bob cannot compare their entire sifted key, because that would reveal it. Instead, they sacrifice a random subset of the sifted key as **check bits**, compare only those bits, and estimate the QBER. If the error rate is above a threshold, they discard the key and run the protocol again. Otherwise, they remove the check bits and keep the rest as the final key.

### Task 6.1: Implement the Check

Implement `check_key(alice_key, bob_key, check_fraction, threshold, rng) -> tuple[str, str, bool]`.

- `alice_key` and `bob_key`: sifted keys of equal length.
- `check_fraction`: fraction of the key to use for checking.
- `threshold`: maximum acceptable error rate on the check bits.
- `rng`: random generator used to choose the check positions.
- Choose `num_check = max(1, int(round(check_fraction * len(alice_key))))` positions without replacement, but never check the entire key.
- Returns `(alice_remaining, bob_remaining, passed)`. The remaining keys have the check bits removed, and `passed` is `True` when the check-bit error rate is at most `threshold`.

> Hint: use `rng.choice(n, size=num_check, replace=False)` to choose positions, then use a set of those positions when building the remaining keys.

In [ ]:
def check_key(
    alice_key: str,
    bob_key: str,
    check_fraction: float,
    threshold: float,
    rng: np.random.Generator,
) -> tuple[str, str, bool]:
    """
    Checks a random subset of the sifted key and removes the check bits.

    Args:
        alice_key: Alice's sifted key.
        bob_key: Bob's sifted key.
        check_fraction: Fraction of the key used for checking.
        threshold: Maximum acceptable error rate on the check bits.
        rng: Random generator used to choose the check positions.

    Returns:
        tuple[str, str, bool]: Alice's remaining key, Bob's remaining key, and
        whether the check passed.
    """
    validate_bits(alice_key, "alice_key")
    validate_bits(bob_key, "bob_key")
    assert len(alice_key) == len(bob_key), "The sifted keys must have the same length."
    assert len(alice_key) >= 2, "The sifted key must have at least two bits."
    assert 0 < check_fraction < 1, "check_fraction must be between 0 and 1."
    assert 0 <= threshold <= 1, "threshold must be between 0 and 1."

    # TODO: choose check positions, compare the check bits, and remove them
    pass

In [ ]:
rng = np.random.default_rng(42)
alice_key = "0101101011"

remaining_a, remaining_b, passed = check_key(
    alice_key, alice_key, 0.5, 0.1, rng
)
assert passed, "An error-free key must pass the check."
assert remaining_a == remaining_b
assert len(remaining_a) == 5, (
    "With check_fraction=0.5, half of the 10 key bits should remain."
)

bob_key = ''.join('1' if bit == '0' else '0' for bit in alice_key)
remaining_a, remaining_b, passed = check_key(
    alice_key, bob_key, 0.5, 0.1, rng
)
assert not passed, "A key with every bit flipped must fail the check."

print("Key checking passed all tests!")

### Task 6.2: Implement the Retry Loop

Implement `secure_bb84(n, rng, eve=False, check_fraction=0.5, threshold=0.05, max_attempts=10) -> tuple[str, str, bool, int]`.

- `n`: number of qubits sent in each attempt.
- `rng`: random generator.
- `eve`: whether Eve intercepts every qubit.
- `check_fraction` and `threshold`: passed to `check_key`.
- `max_attempts`: maximum number of attempts.
- Returns `(alice_key, bob_key, accepted, attempts)`.

On each attempt, generate Alice's bits and bases and Bob's bases. If `eve` is `True`, generate Eve's bases and pass Alice's encoded circuit through `eve_intercept_resend`. Then measure with Bob's bases, sift, and call `check_key`. If the check passes, return the remaining keys with `accepted=True`. Otherwise discard the key and try again. If every attempt fails, return the last attempt's remaining keys with `accepted=False`.

> Hint: keep the last attempt's keys outside the loop so you can return them after the loop. Skip attempts whose sifted key has fewer than two bits.

In [ ]:
def secure_bb84(
    n: int,
    rng: np.random.Generator,
    eve: bool = False,
    check_fraction: float = 0.5,
    threshold: float = 0.05,
    max_attempts: int = 10,
) -> tuple[str, str, bool, int]:
    """
    Runs BB84 with a security check, retrying when the check fails.

    Args:
        n: Number of qubits sent in each attempt.
        rng: Random generator.
        eve: Whether Eve intercepts every qubit.
        check_fraction: Fraction of the sifted key used for checking.
        threshold: Maximum acceptable check-bit error rate.
        max_attempts: Maximum number of attempts.

    Returns:
        tuple[str, str, bool, int]: Alice's key, Bob's key, whether the key was
        accepted, and the number of attempts.
    """
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    assert 0 < check_fraction < 1, "check_fraction must be between 0 and 1."
    assert 0 <= threshold <= 1, "threshold must be between 0 and 1."
    assert isinstance(max_attempts, int) and max_attempts >= 1, (
        "max_attempts must be a positive integer."
    )

    # TODO: run BB84, check the key, and retry if the check fails
    pass

### Task 6.3: Test the Secure Protocol

The cell below first runs the protocol without Eve. The check should pass on the first attempt. It then runs the protocol with Eve. Because Eve disturbs the qubits, the check should fail every attempt, so the key is rejected.

In [ ]:
rng = np.random.default_rng(2024)
alice_key, bob_key, accepted, attempts = secure_bb84(
    n=300,
    rng=rng,
    eve=False,
    check_fraction=0.5,
    threshold=0.05,
    max_attempts=5,
)
assert accepted, "An honest run should accept the key."
assert attempts == 1, "An honest run should pass on the first attempt."
assert alice_key == bob_key
print(f"Honest run: accepted after {attempts} attempt(s), key length = {len(alice_key)}.")

rng = np.random.default_rng(2024)
alice_key, bob_key, accepted, attempts = secure_bb84(
    n=300,
    rng=rng,
    eve=True,
    check_fraction=0.5,
    threshold=0.05,
    max_attempts=10,
)
assert not accepted, "Eve's interference should be detected."
assert attempts == 10, "Every attempt with Eve should fail the check."
print(f"Eavesdropped run: rejected after {attempts} attempts.")